In [2]:
import os
import time
import gc
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: CUDA is NOT available. Running on CPU!")

C:\Users\Vijay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
GPU Total Memory: 6.44 GB


In [3]:
# Dataset Paths
dataset_dir = r"D:\VJ\cattle datasets\clean_cattle_split"
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")
test_dir = os.path.join(dataset_dir, "test")

# High-Speed High-Accuracy Hyperparameters (ConvNeXt-Small @ 384x384)
img_size = 384
batch_size = 16
epochs = 30
warmup_epochs = 3
lr = 1e-4
weight_decay = 0.05
model_name = "convnext_small.fb_in22k_ft_in1k_384"

print(f"Target dataset: {dataset_dir}")
print(f"Model Architecture: {model_name}")
print(f"Image Resolution: {img_size}x{img_size} | Batch Size: {batch_size} | Epochs: {epochs} (Warmup: {warmup_epochs})")

Target dataset: D:\VJ\cattle datasets\clean_cattle_split
Model Architecture: convnext_small.fb_in22k_ft_in1k_384
Image Resolution: 384x384 | Batch Size: 16 | Epochs: 30 (Warmup: 3)


In [4]:
print(f"Loading pre-trained model weights for {model_name} with Stochastic Depth & Dropout...")
# Load model with drop_rate=0.3 & drop_path_rate=0.2 to prevent overfitting
model = timm.create_model(model_name, pretrained=True, num_classes=50, drop_rate=0.3, drop_path_rate=0.2)
data_config = timm.data.resolve_model_data_config(model)
mean = data_config.get('mean', (0.485, 0.456, 0.406))
std = data_config.get('std', (0.229, 0.224, 0.225))
print(f"Model Normalization - Mean: {mean}, Std: {std}")

# Fast Training Augmentations + High-Precision Bicubic Evaluation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Configure Mixup & CutMix
mixup_fn = Mixup(
    mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5,
    mode='batch', label_smoothing=0.1, num_classes=50
)
print("Transforms and Mixup/CutMix configured successfully.")

Loading pre-trained model weights for convnext_small.fb_in22k_ft_in1k_384 with Stochastic Depth & Dropout...


Model Normalization - Mean: (0.485, 0.456, 0.406), Std: (0.229, 0.224, 0.225)
Transforms and Mixup/CutMix configured successfully.


In [5]:
print("Loading dataset splits...")
train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset = ImageFolder(val_dir, transform=eval_transform)
test_dataset = ImageFolder(test_dir, transform=eval_transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")

# DataLoader settings for Windows
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,drop_last=True, num_workers=0, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
print("DataLoaders prepared.")

Loading dataset splits...
Train samples: 3457
Val samples:   740
Test samples:  768
Number of classes: 50
DataLoaders prepared.


In [6]:
# Clear leftover GPU VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Criterion: SoftTargetCrossEntropy for Mixup/CutMix targets
train_criterion = SoftTargetCrossEntropy()
val_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# Sequential LR: Linear Warmup for first 3 epochs, followed by Cosine Annealing
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, main_scheduler],
    milestones=[warmup_epochs]
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
model = model.to(device)
print(f"Model moved to {device}. SoftTargetCrossEntropy, Optimizer, and SequentialLR setup complete.")

Model moved to cuda. SoftTargetCrossEntropy, Optimizer, and SequentialLR setup complete.


C:\Users\Vijay\AppData\Local\Temp\ipykernel_12496\3337478834.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [7]:
# Training and Validation Loop with Mixup Progress Printing
best_val_acc = 0.0
save_path = "best_single_cattle_model.pth"

print("\n=================== STARTING MIXUP & CUTMIX TRAINING ===================")
total_batches = len(train_loader)

for epoch in range(1, epochs + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    total_train = 0

    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n--- Epoch [{epoch:02d}/{epochs:02d}] | LR: {current_lr:.6f} ---")
    
    for step, (images, labels) in enumerate(train_loader, 1):
        images, labels = images.to(device), labels.to(device)
        
        # Apply Mixup & CutMix to batch
        images, mixup_labels = mixup_fn(images, labels)
        
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = train_criterion(outputs, mixup_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

        # Print progress every 20 batches and at batch end
        if step % 20 == 0 or step == total_batches:
            print(f"  Step [{step:03d}/{total_batches:03d}] | Batch Loss: {loss.item():.4f}")

    scheduler.step()
    train_loss = running_loss / total_train

    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = val_criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct_val += preds.eq(labels).sum().item()
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = (correct_val / total_val) * 100.0
    epoch_time = time.time() - start_time

    print(f">> Epoch [{epoch:02d}/{epochs:02d}] Finished in {epoch_time:.1f}s")
    print(f"   Train Loss: {train_loss:.4f}")
    print(f"   Val   Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'classes': train_dataset.classes
        }, save_path)
        print(f"   🔥 Best Model Saved! Peak Val Acc: {val_acc:.2f}%")

print(f"\n=================== TRAINING COMPLETE ===================")
print(f"Peak Validation Accuracy: {best_val_acc:.2f}%")


=================== STARTING MIXUP & CUTMIX TRAINING ===================

--- Epoch [01/30] | LR: 0.000010 ---


C:\Users\Vijay\AppData\Local\Temp\ipykernel_12496\2311665888.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Step [020/216] | Batch Loss: 3.9309
  Step [040/216] | Batch Loss: 3.9438
  Step [060/216] | Batch Loss: 3.9403
  Step [080/216] | Batch Loss: 3.9175
  Step [100/216] | Batch Loss: 3.8564
  Step [120/216] | Batch Loss: 3.9183
  Step [140/216] | Batch Loss: 3.9158
  Step [160/216] | Batch Loss: 3.8322
  Step [180/216] | Batch Loss: 3.7860
  Step [200/216] | Batch Loss: 4.0420
  Step [216/216] | Batch Loss: 3.6080


C:\Users\Vijay\AppData\Local\Temp\ipykernel_12496\2311665888.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


>> Epoch [01/30] Finished in 113.5s
   Train Loss: 3.9007
   Val   Loss: 3.5835 | Val Acc: 14.19%
   🔥 Best Model Saved! Peak Val Acc: 14.19%

--- Epoch [02/30] | LR: 0.000040 ---
  Step [020/216] | Batch Loss: 3.6679
  Step [040/216] | Batch Loss: 3.7368
  Step [060/216] | Batch Loss: 3.8473
  Step [080/216] | Batch Loss: 3.4158
  Step [100/216] | Batch Loss: 3.3947
  Step [120/216] | Batch Loss: 4.0580
  Step [140/216] | Batch Loss: 3.3528
  Step [160/216] | Batch Loss: 3.4306
  Step [180/216] | Batch Loss: 3.4040
  Step [200/216] | Batch Loss: 2.9486
  Step [216/216] | Batch Loss: 2.9515
>> Epoch [02/30] Finished in 99.9s
   Train Loss: 3.5150
   Val   Loss: 2.3021 | Val Acc: 38.11%
   🔥 Best Model Saved! Peak Val Acc: 38.11%

--- Epoch [03/30] | LR: 0.000070 ---
  Step [020/216] | Batch Loss: 3.7593
  Step [040/216] | Batch Loss: 2.9875
  Step [060/216] | Batch Loss: 3.3797
  Step [080/216] | Batch Loss: 2.5381
  Step [100/216] | Batch Loss: 3.2801
  Step [120/216] | Batch Loss: 3.

C:\Users\Vijay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


>> Epoch [03/30] Finished in 99.6s
   Train Loss: 3.0393
   Val   Loss: 1.7121 | Val Acc: 49.86%
   🔥 Best Model Saved! Peak Val Acc: 49.86%

--- Epoch [04/30] | LR: 0.000100 ---
  Step [020/216] | Batch Loss: 2.4076
  Step [040/216] | Batch Loss: 2.8164
  Step [060/216] | Batch Loss: 3.4797
  Step [080/216] | Batch Loss: 3.2590
  Step [100/216] | Batch Loss: 2.9706
  Step [120/216] | Batch Loss: 3.2395
  Step [140/216] | Batch Loss: 2.4521
  Step [160/216] | Batch Loss: 2.9209
  Step [180/216] | Batch Loss: 3.0643
  Step [200/216] | Batch Loss: 2.0347
  Step [216/216] | Batch Loss: 2.3835
>> Epoch [04/30] Finished in 99.1s
   Train Loss: 2.8654
   Val   Loss: 1.5097 | Val Acc: 55.00%
   🔥 Best Model Saved! Peak Val Acc: 55.00%

--- Epoch [05/30] | LR: 0.000100 ---
  Step [020/216] | Batch Loss: 2.9492
  Step [040/216] | Batch Loss: 2.8613
  Step [060/216] | Batch Loss: 2.4072
  Step [080/216] | Batch Loss: 2.4600
  Step [100/216] | Batch Loss: 3.0298
  Step [120/216] | Batch Loss: 2.8

In [8]:
# Final Evaluation on Unseen Test Dataset (with Test-Time Augmentation)
print("\n=================== FINAL TEST EVALUATION (WITH TTA) ===================")
checkpoint = torch.load(save_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

correct_test = 0
total_test = 0
total_test_batches = len(test_loader)

print("Evaluating test set with Horizontal-Flip Test-Time Augmentation...")
with torch.no_grad():
    for step, (images, labels) in enumerate(test_loader, 1):
        images, labels = images.to(device), labels.to(device)
        
        # Test-Time Augmentation (Original + Horizontal Flip)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs_orig = model(images)
            outputs_flip = model(torch.flip(images, dims=[3]))
            outputs = (outputs_orig + outputs_flip) / 2.0
            
        _, preds = outputs.max(1)
        correct_test += preds.eq(labels).sum().item()
        total_test += labels.size(0)
        
        if step % 10 == 0 or step == total_test_batches:
            print(f"  Test Evaluation Progress: [{step:03d}/{total_test_batches:03d}] batches done")

test_acc = (correct_test / total_test) * 100.0
print(f"\n🚀 FINAL UNSEEN TEST ACCURACY (WITH TTA): {test_acc:.2f}%")


=================== FINAL TEST EVALUATION (WITH TTA) ===================


C:\Users\Vijay\AppData\Local\Temp\ipykernel_12496\4190320324.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path)


Evaluating test set with Horizontal-Flip Test-Time Augmentation...


C:\Users\Vijay\AppData\Local\Temp\ipykernel_12496\4190320324.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


  Test Evaluation Progress: [010/048] batches done
  Test Evaluation Progress: [020/048] batches done
  Test Evaluation Progress: [030/048] batches done
  Test Evaluation Progress: [040/048] batches done
  Test Evaluation Progress: [048/048] batches done

🚀 FINAL UNSEEN TEST ACCURACY (WITH TTA): 69.79%
